# Investigating pretrained model

The [pretrained mopadi models are available on huggingface](https://huggingface.co/KatherLab/MoPaDi). But in the repo, the autoencoder and the diffusion model are saved as a combined model checkpoint. The question is now, if it is possible to separate them from another to just use the diffusion model and not the autoencoder.

In [1]:
import os
from huggingface_hub import hf_hub_download
import torch
from collections import Counter, defaultdict

/home/maralampert/micromamba/envs/multiplex_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
token = os.getenv("HF_TOKEN")

In [6]:
# download model files
autoenc_model_path = hf_hub_download(
    repo_id="KatherLab/MoPaDi",
    filename="brca_512_model/autoenc.ckpt",
    cache_dir="/mnt/bulk-mars/maralampert/snickers/rna2wsi/models/mopadi_pretrained/"
)
print(f"Autoencoder's checkpoint downloaded to: {autoenc_model_path}")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Autoencoder's checkpoint downloaded to: /mnt/bulk-mars/maralampert/snickers/rna2wsi/models/mopadi_pretrained/models--KatherLab--MoPaDi/snapshots/5d8e775e24473c5d8f4c0c57fd5c865c3c2a4aab/brca_512_model/autoenc.ckpt


In [18]:
def separate_autoencoder_diffusion(ckpt_path):
    """
    Separates the autoencoder and diffusion model state dictionaries from a checkpoint.

    Args:
        ckpt_path (str): Path to the checkpoint file.

    Returns:
        tuple: A tuple containing the autoencoder state dictionary,
               the diffusion model state dictionary, and any leftover keys.
               Returns (None, None, None) if the checkpoint is invalid.
    """
    try:
        ckpt = torch.load(ckpt_path, map_location="cpu")
        sd = ckpt.get("state_dict", ckpt)  # adapt for PL-style checkpoints
        keys = list(sd.keys())
        print("total keys:", len(keys))

        # Define prefixes for diffusion model
        diffusion_prefixes = [
            "model.time_embed",
            "model.input_blocks",
            "model.output_blocks",
            "model.middle_block",
            "cond_stage_model",
        ]

        # Helper function to check if a key starts with any prefix
        def starts_with_any(key, prefixes):
            return any(key.startswith(prefix) for prefix in prefixes)

        # Functions to categorize keys
        def is_diffusion(key):
            return starts_with_any(key, diffusion_prefixes)

        # Split the keys (exclude EMA for main model)
        main_keys = [k for k in keys if not k.startswith("ema_model.")]
        diff_keys = [k for k in main_keys if is_diffusion(k)]

        # Autoencoder keys are those starting with "model.encoder."
        auto_keys = [k for k in main_keys if k.startswith("model.encoder.")]

        # Verify the split
        print(f"Autoencoder keys (main): {len(auto_keys)}")
        print(f"Diffusion keys (main): {len(diff_keys)}")
        print(f"Leftover keys (main): {len(set(main_keys) - set(auto_keys) - set(diff_keys))}")

        # Print any leftover keys
        leftover = set(main_keys) - set(auto_keys) - set(diff_keys)
        print("First 5 leftover keys (main):", list(leftover)[:5] if leftover else "None")

        # Create new state dictionaries
        autoencoder_sd = {k: sd[k] for k in auto_keys}
        diffusion_sd = {k: sd[k] for k in diff_keys}

        return autoencoder_sd, diffusion_sd, leftover

    except FileNotFoundError:
        print(f"Error: Checkpoint file not found at {ckpt_path}")
        return None, None, None
    except Exception as e:
        print(f"Error loading or processing checkpoint: {e}")
        return None, None, None


In [19]:

# Example Usage:
orig = "/mnt/bulk-mars/maralampert/snickers/rna2wsi/models/mopadi_pretrained/models--KatherLab--MoPaDi/snapshots/5d8e775e24473c5d8f4c0c57fd5c865c3c2a4aab/brca_512_model/autoenc.ckpt"  # Replace with your checkpoint path
autoencoder_sd, diffusion_sd, leftover_keys = separate_autoencoder_diffusion(orig)

if autoencoder_sd and diffusion_sd:
    print("Autoencoder SD keys:", list(autoencoder_sd.keys())[:5])
    print("Diffusion SD keys:", list(diffusion_sd.keys())[:5])

    # Save the separate state dictionaries (optional)
    torch.save(autoencoder_sd, "/mnt/bulk-mars/maralampert/snickers/rna2wsi/models/mopadi_pretrained/autoencoder_split.ckpt")
    torch.save(diffusion_sd, "/mnt/bulk-mars/maralampert/snickers/rna2wsi/models/mopadi_pretrained/diffusion_split.ckpt")

total keys: 1413
Autoencoder keys (main): 116
Diffusion keys (main): 586
Leftover keys (main): 5
First 5 leftover keys (main): ['x_T', 'model.out.0.bias', 'model.out.2.weight', 'model.out.2.bias', 'model.out.0.weight']
Autoencoder SD keys: ['model.encoder.input_blocks.0.0.weight', 'model.encoder.input_blocks.0.0.bias', 'model.encoder.input_blocks.1.0.in_layers.0.weight', 'model.encoder.input_blocks.1.0.in_layers.0.bias', 'model.encoder.input_blocks.1.0.in_layers.2.weight']
Diffusion SD keys: ['model.time_embed.time_embed.0.weight', 'model.time_embed.time_embed.0.bias', 'model.time_embed.time_embed.2.weight', 'model.time_embed.time_embed.2.bias', 'model.input_blocks.0.0.weight']
